In [2]:
import os
import zipfile
import pandas as pd

#Define paths
raw_dir = "../data/raw/"
clean_dir = "../data/cleaned/"
os.makedirs(clean_dir, exist_ok=True)

#STEP 1: Unzip the latest .zip file in raw/
zip_files = [f for f in os.listdir(raw_dir) if f.endswith(".zip")]
if not zip_files:
    raise FileNotFoundError("No .zip file found in data/raw/")

#Use the most recent .zip file
zip_path = os.path.join(raw_dir, zip_files[-1])
print(f"Unzipping: {zip_path}")

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(raw_dir)

print("? Extracted contents to data/raw/")

#STEP 2: Load the extracted CSV
csv_files = [f for f in os.listdir(raw_dir) if f.endswith(".csv")]
if not csv_files:
    raise FileNotFoundError("No .csv file found in data/raw/ after unzipping.")

csv_path = os.path.join(raw_dir, csv_files[0])
df = pd.read_csv(csv_path)

#STEP 3: Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

#STEP 4: Combine year and month into a single 'date' column
df['month'] = df['month'].astype(int)
df['year'] = df['year'].astype(int)
df['date'] = pd.to_datetime(dict(year=df['year'], month=df['month'], day=1))

#STEP 5: Preprocessing and Cleaning

#Drop rows with missing key values
critical_columns = ["arr_flights", "carrier_delay", "late_aircraft_delay"]
df = df.dropna(subset=critical_columns, how="any")

#Remove duplicates
df = df.drop_duplicates()

#Ensure delay columns are numeric and non-negative
delay_cols = ["carrier_delay", "weather_delay", "nas_delay", "security_delay", "late_aircraft_delay"]
df[delay_cols] = df[delay_cols].apply(pd.to_numeric, errors="coerce")  # Convert to float, force errors to NaN
df = df.dropna(subset=delay_cols)  # Drop rows with NaN delays
for col in delay_cols:
    df = df[df[col] >= 0]

#Fill other numeric columns with 0 (optional)
numeric_cols = df.select_dtypes(include=["float", "int"]).columns
df[numeric_cols] = df[numeric_cols].fillna(0)

#STEP 6: Save cleaned version
df.to_csv(os.path.join(clean_dir, "airline_delay_summary_clean.csv"), index=False)

print("? Cleaned data saved to data/cleaned/airline_delay_summary_clean.csv")
df.head()


Unzipping: ../data/raw/ot_delaycause1_DL.zip
? Extracted contents to data/raw/
? Cleaned data saved to data/cleaned/airline_delay_summary_clean.csv


,year,month,carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,weather_ct,...,late_aircraft_ct,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay,date
0,2024,12,MQ,Envoy Air,EVV,"Evansville, IN: Evansville Regional",61.0,9.0,1.52,1.08,...,5.84,0.0,0.0,732.0,47.0,90.0,19.0,0.0,576.0,2024-12-01
1,2024,12,MQ,Envoy Air,EWR,"Newark, NJ: Newark Liberty International",107.0,42.0,6.01,5.89,...,4.94,0.0,0.0,2531.0,335.0,491.0,1251.0,0.0,454.0,2024-12-01
2,2024,12,MQ,Envoy Air,EYW,"Key West, FL: Key West International",169.0,31.0,3.37,0.71,...,15.48,5.0,3.0,1596.0,143.0,52.0,468.0,0.0,933.0,2024-12-01
3,2024,12,MQ,Envoy Air,FAR,"Fargo, ND: Hector International",171.0,35.0,4.64,2.12,...,12.92,2.0,0.0,2428.0,245.0,184.0,575.0,0.0,1424.0,2024-12-01
4,2024,12,MQ,Envoy Air,FSD,"Sioux Falls, SD: Joe Foss Field",69.0,14.0,2.00,2.47,...,4.83,1.0,0.0,720.0,86.0,154.0,191.0,0.0,289.0,2024-12-01
